# Data Science Detective: Can You Outsmart the Algorithm?
**40 minutes | A-Level | JupyterLite | No previous Python required**

**Your mission:** A university wants to offer extra help to students who may struggle with a future assessment. Using **200 fictional student records**, can you make a useful prediction?

You will try **four approaches**: your own judgement, rules you write, a hand-built decision tree, and a **decision tree trained from data**. Then you will compare results.

**How to use this notebook:** Read each short instruction, change only the values marked with a pencil `✏️`, and press **Shift + Enter** to run the next code cell. Do not worry if the code is unfamiliar.

**Key:** `Passed = 1` means the student passed a *later* assessment; `Passed = 0` means they did not. `AssignmentMark` is from an *earlier* piece of work, so it can be used to make a prediction. StudentIDs identify records and must not be used as a predictor.

## 0–5 minutes · Meet the dataset
A data scientist uses observations (**features**) to predict a result (**target**). Our features are attendance percentage, weekly study hours and an earlier assignment mark; our target is the later pass/fail result.

Run the next cell. The computer secretly reserves **150 records for learning** and **50 records for testing**. We will not use the test outcomes to design our classifiers.

In [ ]:
import csv, random, math
from pathlib import Path

# Keep the CSV in the same JupyterLite folder as this notebook.
with open('student_data.csv', newline='') as f:
    students = [{key: (value if key == 'StudentID' else int(value))
                 for key, value in record.items()}
                for record in csv.DictReader(f)]

rng = random.Random(42)
rng.shuffle(students)
train, test = students[:150], students[150:]
challenge = test[:5]  # The five displayed in Task 1
print('Students:', len(students), '| Training:', len(train), '| Test:', len(test))
print('Example row (training data):', train[0])
print('Training pass rate:', round(100*sum(s['Passed'] for s in train)/len(train)), '%')

## Task 1 · Trust your instincts (5–10 minutes)
These are the **five students** whose outcomes you must predict. They come from the unseen test group, and the actual outcomes are hidden for now.

| Student ID | Attendance | Study hours / week | Earlier assignment mark |
|---|---:|---:|---:|
| S138 | 69% | 18 | 93% |
| S118 | 78% | 16 | 58% |
| S012 | 79% | 2 | 70% |
| S068 | 79% | 14 | 32% |
| S089 | 75% | 9 | 43% |

✏️ **Student action 1:** Guess whether each will **pass (`1`)** or **not pass (`0`)**. Replace the five `None` values in the next cell, in table order. Your judgement may be wrong—that is the point of the challenge.

In [ ]:
# Task 1 — record your five predictions in the same order as the table.
my_guesses = [None, None, None, None, None]

if len(my_guesses) == 5 and all(guess in (0, 1) for guess in my_guesses):
    print('Your five predictions are recorded. Continue to Task 2!')
else:
    print('Your turn: replace all five None values with 0 (might need support) or 1 (likely to pass).')


## Task 2 · Write the rules (10–17 minutes)
A **rule-based classifier** applies conditions chosen by a human. For example: *Predict a pass if attendance is at least 70% AND earlier assignment mark is at least 60%.* Study hours are not used in this simple starting model.

✏️ **Student action 2:** Change `ATTENDANCE_LIMIT = 70` to `80`. Rerun the cell. Does it predict **more or fewer** students will pass? Restore `70` before moving on so the original rule can be compared later.

In [ ]:
# ✏️ YOUR TURN: change a number and rerun the cell.
ATTENDANCE_LIMIT = 70
MARK_LIMIT = 60

def rule_predict(student):
    if (student['Attendance'] >= ATTENDANCE_LIMIT and
        student['AssignmentMark'] >= MARK_LIMIT):
        return 1
    return 0

rule_predictions = [rule_predict(s) for s in test]
print('Rule predicts passes:', sum(rule_predictions), 'out of', len(test))
print('First five rule predictions:', rule_predictions[:5])

## 17–21 minutes · Convert your rule into a decision tree
A **decision tree** is a series of yes/no questions. This first tree is written by a human: it is still a **rule-based program**, not machine learning.

```text
                  Attendance >= 70?
                    /          \
                  No            Yes
                  |              |
            Predict 0     AssignmentMark >= 60?
                              /          \
                            No            Yes
                            |              |
                        Predict 0      Predict 1
```

Run the next cell: the two versions should give identical predictions. Notice how a single `and` rule becomes a branching decision process.

In [ ]:
def hand_built_tree(student):
    if student['Attendance'] >= 70:
        if student['AssignmentMark'] >= 60:
            return 1
        else:
            return 0
    else:
        return 0

same = all(rule_predict(s) == hand_built_tree(s) for s in test)
print('Hand-built tree matches the original rule for all 50 test cases:', same)

## 21–29 minutes · Machine learning: let Python choose the questions
Unlike our hand-built rules, a **learned decision tree** selects questions and thresholds by examining *training* data. The algorithm tries splits and prefers those that put pass and fail cases into purer groups.

We measure the mixture using **Gini impurity**: `2 × p × (1 − p)`, where `p` is the fraction who passed. Impurity is **0** for a group containing only passes or only fails; it reaches **0.5** for a 50/50 group.

The next cell contains a short decision-tree learning algorithm written entirely in standard Python. **You do not need to type or understand all of it.** Run it and find the first question the computer learned.

In [ ]:
# No machine-learning libraries: this program learns the questions itself.
FEATURES = ['Attendance', 'StudyHours', 'AssignmentMark']

def gini(rows):
    if not rows:
        return 0
    p = sum(r['Passed'] for r in rows) / len(rows)
    return 2 * p * (1 - p)

def find_best_split(rows, min_leaf=8):
    best = None
    for feature in FEATURES:
        values = sorted(set(r[feature] for r in rows))
        for a, b in zip(values, values[1:]):
            threshold = (a + b) / 2
            left = [r for r in rows if r[feature] <= threshold]
            right = [r for r in rows if r[feature] > threshold]
            if min(len(left), len(right)) < min_leaf:
                continue
            weighted_gini = (len(left)*gini(left) + len(right)*gini(right))/len(rows)
            if best is None or weighted_gini < best[0]:
                best = (weighted_gini, feature, threshold, left, right)
    return best

def build_tree(rows, depth=0, max_depth=3):
    majority = int(sum(r['Passed'] for r in rows) >= len(rows)/2)
    node = {'prediction': majority, 'count': len(rows)}
    if depth >= max_depth or gini(rows) == 0:
        return node
    split = find_best_split(rows)
    if split is None or split[0] >= gini(rows):
        return node
    _, feature, threshold, left, right = split
    node.update(feature=feature, threshold=threshold,
                left=build_tree(left, depth+1, max_depth),
                right=build_tree(right, depth+1, max_depth))
    return node

def tree_predict(student, node):
    while 'feature' in node:
        node = node['left'] if student[node['feature']] <= node['threshold'] else node['right']
    return node['prediction']

def show_tree(node, depth=0):
    prefix = '  ' * depth
    if 'feature' not in node:
        print(prefix + f"Predict {node['prediction']} ({node['count']} examples)")
        return
    print(prefix + f"If {node['feature']} <= {node['threshold']:.1f}:")
    show_tree(node['left'], depth+1)
    print(prefix + 'Else:')
    show_tree(node['right'], depth+1)

# Train with the 150 training records only.
TREE_DEPTH = 3
learned_tree = build_tree(train, max_depth=TREE_DEPTH)
print('First question learned:', learned_tree.get('feature'),
      '<=', learned_tree.get('threshold'))
print('\nThe full learned tree:')
show_tree(learned_tree)

### Student action 3 — Investigate the learned tree
Look at the first question. **Did the computer choose the same feature as you did?** The computer selects thresholds by comparing impurity scores rather than simply copying your rules.

✏️ **Optional experiment:** Change `TREE_DEPTH = 3` to `1`, rerun the previous cell, and look at how the tree becomes simpler. Restore `3` before the evaluation. A deeper tree does not necessarily work better on unseen students.

## 29–36 minutes · Evaluate all three approaches
We now reveal the withheld results. A **confusion matrix** separates correct and incorrect predictions:

| | Predicted pass (`1`) | Predicted not pass (`0`) |
|---|---|---|
| **Actually passed (`1`)** | True positive (TP) | False negative (FN) |
| **Actually did not pass (`0`)** | False positive (FP) | True negative (TN) |

**Accuracy** = `(TP + TN) / number of records`.

*Important:* If the university's priority is helping those likely to struggle, a **false positive** (predict pass, student does not pass) risks *missing someone needing help*. Here a *false negative* may mean additional support is offered to someone who would have passed anyway.

Run the code below to compare the two algorithms on the **same 50 test cases**, and your instinctive guesses on the **five challenge students**. Do not treat the 5-student accuracy as directly comparable with the 50-student scores.

In [ ]:
def evaluate(actual, predicted):
    tp = sum(a == 1 and p == 1 for a, p in zip(actual, predicted))
    fp = sum(a == 0 and p == 1 for a, p in zip(actual, predicted))
    fn = sum(a == 1 and p == 0 for a, p in zip(actual, predicted))
    tn = sum(a == 0 and p == 0 for a, p in zip(actual, predicted))
    return tp, fp, fn, tn, (tp + tn) / len(actual)

def show_evaluation(title, actual, predicted):
    tp, fp, fn, tn, accuracy = evaluate(actual, predicted)
    print('\n' + title)
    print('                     Predicted pass   Predicted not pass')
    print(f'Actually passed           {tp:3d}                 {fn:3d}')
    print(f'Actually not passed       {fp:3d}                 {tn:3d}')
    print(f'Accuracy: {accuracy:.0%}  |  FP: {fp}  |  FN: {fn}')

actual = [s['Passed'] for s in test]
rule_predictions = [rule_predict(s) for s in test]
tree_predictions = [tree_predict(s, learned_tree) for s in test]

show_evaluation('YOUR RULES: 50 unseen students', actual, rule_predictions)
show_evaluation('LEARNED DECISION TREE: same 50 students', actual, tree_predictions)

print('\nREVEAL: THE FIVE CHALLENGE STUDENTS')
for i, student in enumerate(challenge):
    print(student['StudentID'], '| Your guess:', my_guesses[i],
          '| Rule:', rule_predictions[i], '| Learned tree:', tree_predictions[i],
          '| Actual:', student['Passed'])

if len(my_guesses) == 5 and all(x in (0, 1) for x in my_guesses):
    show_evaluation('YOUR INSTINCTS: five students only',
                    [s['Passed'] for s in challenge], my_guesses)
else:
    print('Complete your five guesses in Task 1 to calculate your personal score.')

## Task 4 · Make a recommendation (36–40 minutes)
**Discuss or write one sentence for each:**

1. Which classifier has better accuracy on the 50 test students? Does it also make fewer mistakes that might **miss** students needing support?
2. Did machine learning automatically do better than your hand-written rule? What surprised you?
3. Would you let the university automatically act on these predictions? What other information should it consider?

**What have you learned?** Your instincts become explicit **rules**; rules become a **decision-tree structure**; machine learning chooses its *own* decision-tree thresholds from past examples; finally, unseen data test whether predictions are useful.

**Where next?** Undergraduate **Computer Science, Data Science, AI and Statistics** teach the programming, algorithms and mathematics behind these models. Related careers include **data analyst, data scientist, machine-learning engineer** and **educational researcher**.

**Ethics:** Data are fictional. A predictive label is not a judgement about anyone's ability or potential, and should only inform supportive conversations with appropriate safeguards.